In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
file_path = '/content/drive/MyDrive/Mama/HEK0105410_ET_131224_0624_a.txt'

# Let's inspect the first 10 lines of the file to understand its structure
with open(file_path, 'r') as file:
    for i in range(10):
        line = file.readline()
        if not line:
            break
        print(f"Line {i+1}: {repr(line)}")

Line 1: '00000000\tN\t        TIN: 0016745893         \n'
Line 2: '00000001\tN\t     MENEN ASHENAFI G/WOLD      \n'
Line 3: '00000002\tB\t  ADDIS ABABA   \n'
Line 4: '00000003\tB\tS/C. YEKA W. 08 \n'
Line 5: '00000004\tN\t      H.NO NEW MELYA KU.03      \n'
Line 6: '00000005\tN\t       SHOLA MARKET AREA        \n'
Line 7: '00000006\tB\tTEL:-0930013405 \n'
Line 8: '00000007\tN\t                                \n'
Line 9: '00000008\tN\t        SESSION Z REPORT        \n'
Line 10: '00000009\tN\t\n'


# Task
call:initial_task_agent{user_query: "Separate the raw receipt lines from the main text file \"/content/drive/MyDrive/Mama/HEK0105410_ET_131224_0624_a.txt\" into individual receipt blocks using the 'TIN:' row as the start boundary and 'HEK0105410' as the end boundary, save the resulting list of receipt texts to a single JSON file at \"/content/drive/MyDrive/Mama/separated_receipts.json\", plan a structured extraction and mapping strategy, and initialize a target Pandas DataFrame with the required Amharic and English column names."}

## Separate Receipts and Save to Single JSON File

### Subtask:
Parse the main text file to segment and isolate receipt blocks, then save the list of blocks into a single JSON file.


**Reasoning**:
I will read and parse the text file to separate individual receipt blocks based on the specified start ('TIN:') and end ('HEK0105410') boundary conditions, clean each line, join them, save the list as a JSON file, and verify by displaying the total count and the first block.



In [4]:
import json
import os

file_path = '/content/drive/MyDrive/Mama/HEK0105410_ET_131224_0624_a.txt'
output_path = '/content/drive/MyDrive/Mama/separated_receipts.json'

receipts = []
current_receipt = []
in_receipt = False

# Read all lines from the source file, handling decoding errors robustly
with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
    lines = f.readlines()

for line in lines:
    # Extract the actual text content from the third column
    parts = line.split('\t')
    if len(parts) >= 3:
        content = parts[2].rstrip('\r\n')
    else:
        content = line.rstrip('\r\n')

    # Check if a new receipt starts
    if 'TIN:' in content:
        in_receipt = True
        current_receipt = []

    if in_receipt:
        current_receipt.append(content.strip())
        # Check if the receipt ends
        if 'HEK0105410' in content:
            receipts.append('\n'.join(current_receipt))
            in_receipt = False
            current_receipt = []

# Create directory if it does not exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Write the parsed receipts to a JSON file
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(receipts, f, indent=4)

print(f"Total receipts extracted: {len(receipts)}")
if receipts:
    print("\n--- First Extracted Receipt Block ---")
    print(receipts[0])

Total receipts extracted: 1150

--- First Extracted Receipt Block ---
TIN: 0016745893
MENEN ASHENAFI G/WOLD
ADDIS ABABA
S/C. YEKA W. 08
H.NO NEW MELYA KU.03
SHOLA MARKET AREA
TEL:-0930013405

SESSION Z REPORT

Z:0001
04/04/2023                 09:56

***       SALES REPORT       ***
Customers:                     0
REFUND RECEIPTS                0
Sales:                         0
-  -  -  -
TXBL1                      *0.00
TXBL2                      *0.00
TXBL3                      *0.00
NOTXBL                     *0.00
TAX1 15%                   *0.00
TAX2  0%                   *0.00
TAX3  0%                   *0.00
-  -  -  -
SALES TL                   *0.00
TXBL TL                    *0.00
TAX TL                     *0.00
-  -  -  -
RFD TXBL1                  *0.00
RFD TXBL2                  *0.00
RFD TXBL3                  *0.00
RFD NOTXBL                 *0.00
RFD TAX1 15%               *0.00
RFD TAX2  0%               *0.00
RFD TAX3  0%               *0.00
-  -  -  -
RFD SALES TL

**Reasoning**:

1.   List item
2.   List item


I will initialize the target Pandas DataFrame using Amharic and English column names to capture all metrics, and verify it with a mock structured record as requested.



In [6]:
import pandas as pd

# Define mapping between English and Amharic column names
column_mapping = {
    'TIN': 'የግብር ከፋይ መለያ ቁጥር (TIN)',
    'Taxpayer Name': 'የግብር ከፋይ ስም (Taxpayer Name)',
    'Date': 'ቀን (Date)',
    'Time': 'ሰዓት (Time)',
    'Total Sales': 'ጠቅላላ ሽያጭ (Total Sales)',
    'Total Tax': 'ጠቅላላ ታክስ (Total Tax)',
    'MRC': 'የማሽን መለያ ቁጥር (MRC)',
    'Z-Report Number': 'የዚ-ሪፖርት ቁጥር (Z-Report Number)'
}

# Create a mock structured record representing one parsed receipt
mock_record = {
    column_mapping['TIN']: '0016745893',
    column_mapping['Taxpayer Name']: 'MENEN ASHENAFI G/WOLD',
    column_mapping['Date']: '04/04/2023',
    column_mapping['Time']: '09:56',
    column_mapping['Total Sales']: 0.00,
    column_mapping['Total Tax']: 0.00,
    column_mapping['MRC']: 'HEK0105410',
    column_mapping['Z-Report Number']: '0001'
}

# Initialize target Pandas DataFrame with the mapped bilingual columns directly with the mock record
df = pd.DataFrame([mock_record])

# Display the DataFrame to verify column structure and sample data
print("DataFrame initialized successfully!")
print(df.info())
display(df)

DataFrame initialized successfully!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   የግብር ከፋይ መለያ ቁጥር (TIN)         1 non-null      object 
 1   የግብር ከፋይ ስም (Taxpayer Name)    1 non-null      object 
 2   ቀን (Date)                      1 non-null      object 
 3   ሰዓት (Time)                     1 non-null      object 
 4   ጠቅላላ ሽያጭ (Total Sales)         1 non-null      float64
 5   ጠቅላላ ታክስ (Total Tax)           1 non-null      float64
 6   የማሽን መለያ ቁጥር (MRC)             1 non-null      object 
 7   የዚ-ሪፖርት ቁጥር (Z-Report Number)  1 non-null      object 
dtypes: float64(2), object(6)
memory usage: 196.0+ bytes
None


,የግብር ከፋይ መለያ ቁጥር (TIN),የግብር ከፋይ ስም (Taxpayer Name),ቀን (Date),ሰዓት (Time),ጠቅላላ ሽያጭ (Total Sales),ጠቅላላ ታክስ (Total Tax),የማሽን መለያ ቁጥር (MRC),የዚ-ሪፖርት ቁጥር (Z-Report Number)
0,0016745893,MENEN ASHENAFI G/WOLD,04/04/2023,09:56,0.0,0.0,HEK0105410,0001


## Plan Structured Extraction and Value Mapping (Step 2)

### Subtask:
Develop a detailed parsing strategy and schema mapping to transform the unstructured receipt text blocks into structured data fields.


### Receipt Extraction & Mapping Strategy

To transform the unstructured receipt blocks into structured database columns, we will utilize regular expressions and line-by-line parsing strategies. Here is the blueprint:

#### 1. Key Fields Identification
Based on the sample receipt block, we identify key fields and target columns:
*   **Header Info**: TIN (`TIN: 0016745893`), Taxpayer Name (`MENEN ASHENAFI G/WOLD`), Address/Branch details.
*   **Metadata**:
    *   Z Report Number (`Z:0001`)
    *   Dates and Times (`04/04/2023 09:56`)
    *   Machine Registration Code / MRC (`HEK0105410`)
*   **Financial Accumulators** (e.g., `TXBL1`, `TAX1`, `SALES TL`, `ACCU.FS`)

#### 2. Regular Expression & Extraction Rules
We can extract specific lines or patterns using target matching:
*   **TIN**: `r"TIN:\s*(\d+)"`
*   **Z Report**: `r"Z:(\d+)"`
*   **MRC**: The last non-empty line matching the serial pattern `r"(HEK\d+)"`
*   **Date/Time**: `r"(\d{2}/\d{2}/\d{4})\s+(\d{2}:\d{2})"`
*   **Financial Metrics**: Line-based key-value splitting on spacing or `*` indicator (e.g., `SALES TL` -> price).

#### 3. Standardized Target Dictionary Layout
Each receipt block will be mapped to a standardized Python dictionary structure:
```python
{
    'tin': str,
    'taxpayer_name': str,
    'mrc': str,
    'z_report': str,
    'date_time': str,
    'sales_total': float,
    'tax_total': float,
    'accu_fs': str
}
```

#### 4. Handling Edge Cases & Varied Formats
*   **Empty fields**: Fallback to `None` or `0.0` for numeric values.
*   **Missing dates**: Look for alternative date strings using multiple regex fallback definitions.
*   **Text Cleanup**: Strip whitespace and trailing non-printable characters before parsing.

## Initialize Target Pandas DataFrame (Step 3)

### Subtask:
Establish a target Pandas DataFrame with the required Amharic and English column names, and test it using a mock structured record.


## Summary:

### Q&A

* **How were the raw receipt lines separated and saved?**
  The raw source file `/content/drive/MyDrive/Mama/HEK0105410_ET_131224_0624_a.txt` was parsed by identifying `'TIN:'` as the starting boundary of a receipt and `'HEK0105410'` as the ending boundary. The isolated receipt text blocks were then consolidated and saved as a JSON array to `/content/drive/MyDrive/Mama/separated_receipts.json`.

* **What strategy was planned for structured extraction and mapping?**
  A robust strategy using regular expressions was designed to target key fields (TIN, Taxpayer Name, Date/Time, Z Report Number, MRC, and financial metrics like Sales Total and Tax Total) while accounting for empty fields, missing values, and formatting discrepancies.

* **How was the target Pandas DataFrame initialized?**
  A target Pandas DataFrame was initialized using a bilingual (Amharic and English) column schema mapping to accommodate the extracted metrics. The structure was verified using a mock record to ensure compatibility and correct data types.

---

### Data Analysis Key Findings

* **Total Receipts Extracted**: A total of **1,150 receipt blocks** were successfully isolated, segmented, and extracted from the raw text file.
* **Encoding & Robust Parsing**: The source file contained non-UTF-8 bytes, requiring robust error handling (`errors='replace'`) with UTF-8 encoding during file read operations to prevent execution failures.
* **Bilingual Schema Structure**: The target DataFrame was defined with 8 clean bilingual columns:
  * **TIN**: `የግብር ከፋይ መለያ ቁጥር (TIN)`
  * **Taxpayer Name**: `የግብር ከፋይ ስም (Taxpayer Name)`
  * **Date**: `ቀን (Date)`
  * **Time**: `ሰዓት (Time)`
  * **Total Sales**: `ጠቅላላ ሽያጭ (Total Sales)`
  * **Total Tax**: `ጠቅላላ ታክስ (Total Tax)`
  * **MRC**: `የማሽን መለያ ቁጥር (MRC)`
  * **Z-Report Number**: `የዚ-ሪፖርት ቁጥር (Z-Report Number)`

---

### Insights or Next Steps

* **Next Step**: Implement the planned regular expression parser on the 1,150 raw receipt blocks stored in `/content/drive/MyDrive/Mama/separated_receipts.json` and programmatically populate the initialized bilingual Pandas DataFrame.
* **Data Quality Check**: After batch parsing, perform validation checks on the parsed numeric fields (`Total Sales` and `Total Tax`) to detect and handle any potential scanning or formatting anomalies in the raw text.


# Task
call:initial_task_agent{user_query: "Initialize the exact target Pandas DataFrame using only the specified columns ['ተ. ቁ.', 'የተሸጠው የእቃ መጠሪያ', 'የሥሪት ሃገር', 'Brand Name', 'መለኪያ', 'ብዛት ', 'የአንዱ አማካይ ግዢ ዋጋ UNIT', 'የአንዱ ሽያጭ ዋጋ ከተእታ በፊት', 'ጠቅላላ ሽያጭ ዋጋ ከተእታ በፊት', 'ተ እ ታ', 'ጠቅላላ ዋጋ ታክስ ጨምሮ', 'የሽያጭ  ደረሰኝ ቁጥር (FS No.)', 'ሽያጭ የተከናወነበት ቀን /ወር/', 'የሽያጭ መመዝገቢያ  መለያ ኮድ /MRC/'], analyze and present the structural breakdown of the raw receipt text types from \"/content/drive/MyDrive/Mama/separated_receipts.json\", and propose a detailed mapping strategy to convert these texts to the target columns before performing the extraction."}

## Adjust DataFrame to Exact Target Columns

### Subtask:
Initialize the target Pandas DataFrame using the exact 14 columns specified in the user's task description.


**Reasoning**:
I will initialize the target Pandas DataFrame using the exact 14 columns specified in the instructions and verify its structure.



In [7]:
import pandas as pd

# Define the exact 14 columns specified in the instructions
target_columns = [
    'ተ. ቁ.',
    'የተሸጠው የእቃ መጠሪያ',
    'የሥሪት ሃገር',
    'Brand Name',
    'መለኪያ',
    'ብዛት ',
    'የአንዱ አማካይ ግዢ ዋጋ UNIT',
    'የአንዱ ሽያጭ ዋጋ ከተእታ በፊት',
    'ጠቅላላ ሽያጭ ዋጋ ከተእታ በፊት',
    'ተ እ ታ',
    'ጠቅላላ ዋጋ ታክስ ጨምሮ',
    'የሽያጭ  ደረሰኝ ቁጥር (FS No.)',
    'ሽያጭ የተከናወነበት ቀን /ወር/',
    'የሽያጭ መመዝገቢያ  መለያ ኮድ /MRC/'
]

# Initialize an empty Pandas DataFrame with these exact columns
target_df = pd.DataFrame(columns=target_columns)

# Display DataFrame info and structure to verify
print("Target DataFrame successfully initialized!")
print(target_df.info())
display(target_df.head())


Target DataFrame successfully initialized!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   ተ. ቁ.                      0 non-null      object
 1   የተሸጠው የእቃ መጠሪያ             0 non-null      object
 2   የሥሪት ሃገር                   0 non-null      object
 3   Brand Name                 0 non-null      object
 4   መለኪያ                       0 non-null      object
 5   ብዛት                        0 non-null      object
 6   የአንዱ አማካይ ግዢ ዋጋ UNIT       0 non-null      object
 7   የአንዱ ሽያጭ ዋጋ ከተእታ በፊት       0 non-null      object
 8   ጠቅላላ ሽያጭ ዋጋ ከተእታ በፊት       0 non-null      object
 9   ተ እ ታ                      0 non-null      object
 10  ጠቅላላ ዋጋ ታክስ ጨምሮ            0 non-null      object
 11  የሽያጭ  ደረሰኝ ቁጥር (FS No.)    0 non-null      object
 12  ሽያጭ የተከናወነበት ቀን /ወር/       0 non-null      object
 13  የሽያጭ መመዝገቢያ  መለያ ኮድ /MRC/  0 non-n

,ተ. ቁ.,የተሸጠው የእቃ መጠሪያ,የሥሪት ሃገር,Brand Name,መለኪያ,ብዛት,የአንዱ አማካይ ግዢ ዋጋ UNIT,የአንዱ ሽያጭ ዋጋ ከተእታ በፊት,ጠቅላላ ሽያጭ ዋጋ ከተእታ በፊት,ተ እ ታ,ጠቅላላ ዋጋ ታክስ ጨምሮ,የሽያጭ ደረሰኝ ቁጥር (FS No.),ሽያጭ የተከናወነበት ቀን /ወር/,የሽያጭ መመዝገቢያ መለያ ኮድ /MRC/


**Reasoning**:
Analyze the structural breakdown of the raw receipt text types from `/content/drive/MyDrive/Mama/separated_receipts.json` by inspecting a sample of different receipts.



In [8]:
import json

with open('/content/drive/MyDrive/Mama/separated_receipts.json', 'r', encoding='utf-8') as f:
    receipts = json.load(f)

print(f"Total loaded receipts: {len(receipts)}")

# Let's inspect a few distinct receipts to analyze different structural formats
# (e.g., Z reports, active sales receipts, empty or null receipts)
for i in [0, 50, 100, 500]:
    if i < len(receipts):
        print(f"\n--- Receipt #{i} Sample (First 15 lines) ---")
        lines = receipts[i].split('\n')
        for line in lines[:15]:
            print(line)


Total loaded receipts: 1150

--- Receipt #0 Sample (First 15 lines) ---
TIN: 0016745893
MENEN ASHENAFI G/WOLD
ADDIS ABABA
S/C. YEKA W. 08
H.NO NEW MELYA KU.03
SHOLA MARKET AREA
TEL:-0930013405

SESSION Z REPORT

Z:0001
04/04/2023                 09:56

***       SALES REPORT       ***
Customers:                     0

--- Receipt #50 Sample (First 15 lines) ---
TIN: 0016745893
MENEN ASHENAFI G/WOLD
YEKET TRADING
ADDIS ABABA, S/C. YEKA W. 08
H.NO NEW MELYA KU.03
SHOLA MARKET AREA
TEL:-0930013405

FS No.00000032
21/04/2023                 13:34
Socket/ plug 16A 3ti      *86.95
SUBTOTAL:                 *86.95
-  -  -  -
TXBL1                     *86.95
TAX1 15%                  *13.04

--- Receipt #100 Sample (First 15 lines) ---
TIN: 0016745893
MENEN ASHENAFI G/WOLD
YEKET TRADING
ADDIS ABABA, S/C. YEKA W. 08
H.NO NEW MELYA KU.03
SHOLA MARKET AREA
TEL:-0930013405

FS No.00000060
09/05/2023                 11:55
Switch  16A songru        *69.56
SUBTOTAL:                 *69.56
-  -  -  -


### Receipt Types Structural Analysis & Mapping Strategy

#### 1. Structural Breakdown of Raw Receipt Text Types
Upon analyzing the loaded receipts, we find two primary structural formats:
- **Type A: SESSION Z REPORTS (e.g., Receipt #0)**
  - Contains `SESSION Z REPORT` header.
  - Key fields are `Z:XXXX` (Z-Report number) instead of `FS No.`
  - These reports contain summarized metadata and cumulative daily totals rather than specific itemized sales. Since they represent session summaries, they may not map to item-level details but are critical for MRC and date reference points.
- **Type B: SALES RECEIPTS (e.g., Receipts #50, #100, #500)**
  - Contains an explicit `FS No.XXXXXXXX` identifier (Fiscal Receipt Number).
  - Contains date, time, single or multiple item purchase lines (e.g., `Switch 16A songru *69.56`), followed by financial aggregations (`SUBTOTAL`, `TXBL1`, `TAX1 15%`, `TOTAL`).
  - These are the primary sources for item-level transactions.

#### 2. Column Mapping Strategy to Target DataFrame
Here is how we will parse, extract, and map values from each text block into our exact target columns:

1. **`ተ. ቁ.` (Serial Number)**:
   - Calculated dynamically during dataframe construction representing sequential row numbers.
2. **`የተሸጠው የእቃ መጠሪያ` (Item Name)**:
   - Extracted from the lines between the date-time row and the `SUBTOTAL:` row in a sales receipt (e.g., `Socket/ plug 16A 3ti`). We strip trailing asterisks or prices.
3. **`የሥሪት ሃገር` (Country of Origin)**:
   - Defaults to empty or `None` as it is not explicitly present in the receipt headers.
4. **`Brand Name` (Brand)**:
   - Extracted from the item name line if a brand word exists (e.g., `songru`), otherwise left empty/inferred.
5. **`መለኪያ` (Unit of Measure)**:
   - Extracted or parsed from item description (e.g., `met`, `pcs`), otherwise defaulted to standard default values or empty.
6. **`ብዛት ` (Quantity)**:
   - Parsed from item rows if multiple quantities exist, or defaults to `1` when an item is directly listed with a single price.
7. **`የአንዱ አማካይ ግዢ ዋጋ UNIT` (Average Purchase Price Unit)**:
   - Set as null/empty as this is a sales receipt and purchase cost is typically omitted.
8. **`የአንዱ ሽያጭ ዋጋ ከተእታ በፊት` (Unit Sales Price Before VAT)**:
   - Equal to `ጠቅላላ ሽያጭ ዋጋ ከተእታ በፊት` / `ብዛት `.
9. **`ጠቅላላ ሽያጭ ዋጋ ከተእታ በፊት` (Total Sales Price Before VAT)**:
   - Extracted from `TXBL1` value (e.g., `86.95` from `TXBL1 *86.95`).
10. **`ተ እ ታ` (VAT)**:
    - Extracted from the `TAX1` row (e.g., `13.04` from `TAX1 15% *13.04`).
11. **`ጠቅላላ ዋጋ ታክስ ጨምሮ` (Total Value Including Tax)**:
    - Extracted from `TOTAL:` row (e.g., `99.99` from `TOTAL: *99.99`).
12. **`የሽያጭ  ደረሰኝ ቁጥር (FS No.)` (Sales Receipt Number)**:
    - Extracted via regex `FS No.(\d+)` for Sales Receipts, or `Z:(\d+)` for Z Reports.
13. **`ሽያጭ የተከናወነበት ቀን /ወር/` (Date of Sale)**:
    - Extracted via regex `(\d{2}/\d{2}/\d{4})` found immediately below the FS No. line.
14. **`የሽያጭ መመዝገቢያ  መለያ ኮድ /MRC/` (Machine Registration Code / MRC)**:
    - Extracted via regex `HEK\d+` near the bottom of each block.

## Analyze and Propose Receipt Mapping Strategy

### Subtask:
Analyze the text structure of different receipt types and formulate a robust strategy to map extracted text segments to the 14 target columns.


**Reasoning**:
Categorize receipts into Z reports and Sales receipts, print structural counts, analyze sample features, and formulate a mapping strategy for the 14 columns.



In [9]:
import re
import json

with open('/content/drive/MyDrive/Mama/separated_receipts.json', 'r', encoding='utf-8') as f:
    receipts = json.load(f)

z_reports = []
sales_receipts = []

for r in receipts:
    if 'Z:' in r or 'SESSION Z REPORT' in r:
        z_reports.append(r)
    elif 'FS No.' in r:
        sales_receipts.append(r)
    else:
        sales_receipts.append(r) # Fallback

print(f"Total separated receipts: {len(receipts)}")
print(f"Number of Z Reports: {len(z_reports)}")
print(f"Number of Sales Receipts: {len(sales_receipts)}")

# Let's inspect a typical sales receipt to outline structural mapping
if sales_receipts:
    print("\n--- Sample Sales Receipt Structure ---")
    print("\n".join(sales_receipts[0].split('\n')[:25]))

# Print detailed programmatic mapping strategy for 14 columns
print("""
========================================================================
14 Target Column Mapping Strategy:
========================================================================
1. ተ. ቁ. (Serial No.): Row-index generator during item loop.
2. የተሸጠው የእቃ መጠሪያ (Item Name): Match first line following the Date/Time pattern, up to the '*price' format.
3. የሥሪት ሃገር (Country of Origin): Leave as None/Empty (not present in raw text).
4. Brand Name: Leave as None/Empty (not present in raw text).
5. መለኪያ (Unit): Extract from item line if present (e.g. 'met', 'pcs') or leave empty.
6. ብዛት (Quantity): Extracted if 'QTY' or quantity multipliers exist, otherwise defaults to 1.
7. የአንዱ አማካይ ግዢ ዋጋ UNIT: Default to None or derive if possible.
8. የአንዱ ሽያጭ ዋጋ ከተእታ በፊት (Unit Price excl. VAT): Calculated as 'ጠቅላላ ሽያጭ ዋጋ ከተእታ በፊት' / 'ብዛት'.
9. ጠቅላላ ሽያጭ ዋጋ ከተእታ በፊት (Total Sales excl. VAT): Extract from 'TXBL1' or corresponding TXBL line.
10. ተ እ ታ (VAT): Extract from 'TAX1' / 'TAX1 15%' line matching '*value'.
11. ጠቅላላ ዋጋ ታክስ ጨምሮ (Total Price incl. VAT): Extract from 'TOTAL:' or 'SUBTOTAL:' line matching '*value'.
12. የሽያጭ ደረሰኝ ቁጥር (FS No.): Extract using regex r'FS No.(\d+)'.
13. ሽያጭ የተከናወነበት ቀን /ወር/ (Date): Extract using regex r'(\d{2}/\d{2}/\d{4})'.
14. የሽያጭ መመዝገቢያ መለያ ኮድ /MRC/ (MRC Code): Extract from last line matching r'(HEK\d+)'.
========================================================================
""")

Total separated receipts: 1150
Number of Z Reports: 724
Number of Sales Receipts: 426

--- Sample Sales Receipt Structure ---
TIN: 0016745893
MENEN ASHENAFI G/WOLD
YEKET TRADING
ADDIS ABABA, S/C. YEKA W. 08
H.NO NEW MELYA KU.03
SHOLA MARKET AREA
TEL:-0930013405

FS No.00000001
08/04/2023                 09:09
Electirc wire ilslc        *4.34
SUBTOTAL:                  *4.34
-  -  -  -
TXBL1                      *4.34
TAX1 15%                   *0.65
-  -  -  -
TOTAL:     *4.99
CASH                       *4.99
ITEM#                          1
E R C A
HEK0105410

14 Target Column Mapping Strategy:
1. ተ. ቁ. (Serial No.): Row-index generator during item loop.
2. የተሸጠው የእቃ መጠሪያ (Item Name): Match first line following the Date/Time pattern, up to the '*price' format.
3. የሥሪት ሃገር (Country of Origin): Leave as None/Empty (not present in raw text).
4. Brand Name: Leave as None/Empty (not present in raw text).
5. መለኪያ (Unit): Extract from item line if present (e.g. 'met', 'pcs') or leave empty.


<>:43: SyntaxWarning: invalid escape sequence '\d'
<>:43: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_1829/4034388005.py:43: SyntaxWarning: invalid escape sequence '\d'
  12. የሽያጭ ደረሰኝ ቁጥር (FS No.): Extract using regex r'FS No.(\d+)'.


**Reasoning**:
The previous code block triggered SyntaxWarning due to raw regex strings with backslashes used inside a non-raw triple-quoted string block. To fix this, I will use raw strings (r""") for the print statement to avoid any backslash escape sequence warnings.



In [10]:
import re
import json

with open('/content/drive/MyDrive/Mama/separated_receipts.json', 'r', encoding='utf-8') as f:
    receipts = json.load(f)

z_reports = []
sales_receipts = []

for r in receipts:
    if 'Z:' in r or 'SESSION Z REPORT' in r:
        z_reports.append(r)
    elif 'FS No.' in r:
        sales_receipts.append(r)
    else:
        sales_receipts.append(r) # Fallback

print(f"Total separated receipts: {len(receipts)}")
print(f"Number of Z Reports: {len(z_reports)}")
print(f"Number of Sales Receipts: {len(sales_receipts)}")

# Let's inspect a typical sales receipt to outline structural mapping
if sales_receipts:
    print("\n--- Sample Sales Receipt Structure ---")
    print("\n".join(sales_receipts[0].split('\n')[:25]))

# Print detailed programmatic mapping strategy using a raw triple-quoted string
print(r"""
========================================================================
14 Target Column Mapping Strategy:
========================================================================
1. ተ. ቁ. (Serial No.): Row-index generator during item loop.
2. የተሸጠው የእቃ መጠሪያ (Item Name): Match first line following the Date/Time pattern, up to the '*price' format.
3. የሥሪት ሃገር (Country of Origin): Leave as None/Empty (not present in raw text).
4. Brand Name: Leave as None/Empty (not present in raw text).
5. መለኪያ (Unit): Extract from item line if present (e.g. 'met', 'pcs') or leave empty.
6. ብዛት (Quantity): Extracted if 'QTY' or quantity multipliers exist, otherwise defaults to 1.
7. የአንዱ አማካይ ግዢ ዋጋ UNIT: Default to None or derive if possible.
8. የአንዱ ሽያጭ ዋጋ ከተእታ በፊት (Unit Price excl. VAT): Calculated as 'ጠቅላላ ሽያጭ ዋጋ ከተእታ በፊት' / 'ብዛት'.
9. ጠቅላላ ሽያጭ ዋጋ ከተእታ በፊት (Total Sales excl. VAT): Extract from 'TXBL1' or corresponding TXBL line.
10. ተ እ ታ (VAT): Extract from 'TAX1' / 'TAX1 15%' line matching '*value'.
11. ጠቅላላ ዋጋ ታክስ ጨምሮ (Total Price incl. VAT): Extract from 'TOTAL:' or 'SUBTOTAL:' line matching '*value'.
12. የሽያጭ ደረሰኝ ቁጥር (FS No.): Extract using regex r'FS No.(\d+)'.
13. ሽያጭ የተከናወነበት ቀን /ወር/ (Date): Extract using regex r'(\d{2}/\d{2}/\d{4})'.
14. የሽያጭ መመዝገቢያ መለያ ኮድ /MRC/ (MRC Code): Extract from last line matching r'(HEK\d+)'.
========================================================================
""")

Total separated receipts: 1150
Number of Z Reports: 724
Number of Sales Receipts: 426

--- Sample Sales Receipt Structure ---
TIN: 0016745893
MENEN ASHENAFI G/WOLD
YEKET TRADING
ADDIS ABABA, S/C. YEKA W. 08
H.NO NEW MELYA KU.03
SHOLA MARKET AREA
TEL:-0930013405

FS No.00000001
08/04/2023                 09:09
Electirc wire ilslc        *4.34
SUBTOTAL:                  *4.34
-  -  -  -
TXBL1                      *4.34
TAX1 15%                   *0.65
-  -  -  -
TOTAL:     *4.99
CASH                       *4.99
ITEM#                          1
E R C A
HEK0105410

14 Target Column Mapping Strategy:
1. ተ. ቁ. (Serial No.): Row-index generator during item loop.
2. የተሸጠው የእቃ መጠሪያ (Item Name): Match first line following the Date/Time pattern, up to the '*price' format.
3. የሥሪት ሃገር (Country of Origin): Leave as None/Empty (not present in raw text).
4. Brand Name: Leave as None/Empty (not present in raw text).
5. መለኪያ (Unit): Extract from item line if present (e.g. 'met', 'pcs') or leave empty.


## Final Task

### Subtask:
Present the empty target DataFrame template, summarize the structural parsing proposal, and ask the user for feedback before writing the extraction logic.


## Summary:

### Q&A

* **How is the target Pandas DataFrame structured?**
  An empty target Pandas DataFrame has been initialized with exactly 14 columns matching the specified schema:
  `['ተ. ቁ.', 'የተሸጠው የእቃ መጠሪያ', 'የሥሪት ሃገር', 'Brand Name', 'መለኪያ', 'ብዛት ', 'የአንዱ አማካይ ግዢ ዋጋ UNIT', 'የአንዱ ሽያጭ ዋጋ ከተእታ በፊት', 'ጠቅላላ ሽያጭ ዋጋ ከተእታ በፊት', 'ተ እ ታ', 'ጠቅላላ ዋጋ ታክስ ጨምሮ', 'የሽያጭ  ደረሰኝ ቁጥር (FS No.)', 'ሽያጭ የተከናወነበት ቀን /ወር/', 'የሽያጭ መመዝገቢያ  መለያ ኮድ /MRC/']`.

* **What are the structural raw receipt text types found in the dataset?**
  The dataset consists of 1,150 total records split into two primary formats:
  * **Z Reports (724 receipts):** Contain session totals, daily cumulative values, and a `SESSION Z REPORT` or `Z:` prefix instead of individual transactional items.
  * **Sales Receipts (426 receipts):** Contain itemized sales transactions, explicit item descriptions, price breakdowns, an `FS No.`, and a Machine Registration Code (MRC).

---

### Data Analysis Key Findings

* **Unstructured Dataset Volume:** Out of 1,150 extracted blocks in `separated_receipts.json`, 63% (724 blocks) are daily/session Z-Reports, while 37% (426 blocks) are standard customer Sales Receipts.
* **Extraction Targets:**
  * **FS No. and Dates:** Readily parseable from Sales Receipts via regex patterns matching `FS No.(\d+)` and standard date formatting `(\d{2}/\d{2}/\d{4})`.
  * **Machine Registration Code (MRC):** Consistently located near the footer of the receipt matching the prefix pattern `HEK\d+`.
  * **Financial Math Consistency:** Taxable totals (`TXBL1`), VAT rates (`TAX1 15%`), and grand totals (`TOTAL:`) follow a distinct asterisk price delimiter (e.g., `*86.95`), making it highly suitable for reliable regex parsing.
  * **Omitted Columns:** Columns like `"የሥሪት ሃገር"` (Country of Origin) and `"የአንዱ አማካይ ግዢ ዋጋ UNIT"` (Average Purchase Unit Cost) are absent from raw customer receipts and will default to empty values.

---

### Insights or Next Steps

* **Execute the Regex-Based Parsing Pipeline:** Programmatically process the 426 Sales Receipts using targeted regular expressions to populate the initialized 14-column DataFrame, ignoring or handling the 724 aggregate Z-Reports separately depending on audit needs.
* **Validate Calculated Fields:** Ensure fallback logic is established for items missing explicit multipliers (defaulting quantity to 1) and dynamically compute unit prices excluding VAT to avoid structural data mismatching.
